Imports

In [48]:
import pandas as pd
import numpy as np
import time

Import data

In [49]:
user_features = pd.read_parquet("processed/user_features.parquet")

print("Shape:", user_features.shape)
print("\nColumns:")
print(user_features.columns.tolist())

user_features.head()

Shape: (206209, 21)

Columns:
['user_id', 'alcohol', 'babies', 'bakery', 'beverages', 'breakfast', 'bulk', 'canned_goods', 'dairy_eggs', 'deli', 'dry_goods_pasta', 'frozen', 'household', 'international', 'meat_seafood', 'other', 'pantry', 'personal_care', 'pets', 'produce', 'snacks']


,user_id,alcohol,babies,bakery,beverages,breakfast,bulk,canned_goods,dairy_eggs,deli,...,frozen,household,international,meat_seafood,other,pantry,personal_care,pets,produce,snacks
0,1,0.000000,0.0,0.000000,0.111111,0.055556,0.0,0.000000,0.277778,0.000000,...,0.000000,0.055556,0.000000,0.000000,0.0,0.055556,0.000000,0.0,0.222222,0.222222
1,2,0.000000,0.0,0.019608,0.058824,0.029412,0.0,0.009804,0.176471,0.117647,...,0.117647,0.000000,0.019608,0.009804,0.0,0.068627,0.009804,0.0,0.156863,0.205882
2,3,0.000000,0.0,0.000000,0.060606,0.000000,0.0,0.000000,0.181818,0.030303,...,0.090909,0.030303,0.000000,0.000000,0.0,0.060606,0.000000,0.0,0.363636,0.151515
3,4,0.117647,0.0,0.117647,0.176471,0.000000,0.0,0.058824,0.000000,0.117647,...,0.117647,0.117647,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.117647,0.058824
4,5,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.043478,0.173913,0.043478,...,0.086957,0.000000,0.043478,0.000000,0.0,0.086957,0.000000,0.0,0.434783,0.043478


Clean Data for processing

In [50]:
user_ids = user_features["user_id"].to_numpy()

feature_columns = [
    column
    for column in user_features.columns
    if column != "user_id"
]

X = user_features[feature_columns].to_numpy(dtype=float)

print("Number of users:", X.shape[0])
print("Number of features:", X.shape[1])
print("Missing values:", np.isnan(X).sum())
print("Infinite values:", np.isinf(X).sum())

print("\nFirst 10 row sums:")
print(X.sum(axis=1)[:10])

print("\nFeature minimum:", X.min())
print("Feature maximum:", X.max())

Number of users: 206209
Number of features: 20
Missing values: 0
Infinite values: 0

First 10 row sums:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

Feature minimum: 0.0
Feature maximum: 1.0


BFR Parameters

In [51]:
K = 4
CHUNK_SIZE = 20000
MAHALANOBIS_THRESHOLD = 4.0
RANDOM_STATE = 42

BFR Summary Class

In [52]:
class BFRSummary:

    def __init__(self, points):

        points = np.asarray(points, dtype=float)

        if points.ndim == 1:
            points = points.reshape(1, -1)

        self.N = points.shape[0]

        self.SUM = np.sum(
            points,
            axis=0
        )

        self.SUMSQ = np.sum(
            points ** 2,
            axis=0
        )


    def centroid(self):

        return self.SUM / self.N


    def variance(self):

        variance = (
            self.SUMSQ / self.N
            - (self.SUM / self.N) ** 2
        )

        # Protect against tiny negative numbers
        # caused by floating-point arithmetic.
        return np.maximum(
            variance,
            0.0
        )


    def std(self):

        return np.sqrt(
            self.variance()
        )


    def add_point(self, point):

        point = np.asarray(
            point,
            dtype=float
        )

        self.N += 1

        self.SUM += point

        self.SUMSQ += point ** 2


    def merge(self, other):

        self.N += other.N

        self.SUM += other.SUM

        self.SUMSQ += other.SUMSQ

Test Class

In [53]:
test_points = np.array([
    [5.0,  1.0],
    [6.0, -2.0],
    [7.0,  0.0]
])

test_summary = BFRSummary(
    test_points
)

print("N:")
print(test_summary.N)

print("\nSUM:")
print(test_summary.SUM)

print("\nSUMSQ:")
print(test_summary.SUMSQ)

print("\nCentroid:")
print(test_summary.centroid())

print("\nVariance:")
print(test_summary.variance())

print("\nStandard deviation:")
print(test_summary.std())

N:
3

SUM:
[18. -1.]

SUMSQ:
[110.   5.]

Centroid:
[ 6.         -0.33333333]

Variance:
[0.66666667 1.55555556]

Standard deviation:
[0.81649658 1.24721913]


test algorithm

In [54]:
def mahalanobis_from_values(point, centroid, std):
    point = np.asarray(point, dtype=float)
    centroid = np.asarray(centroid, dtype=float)
    std = np.asarray(std, dtype=float)

    return np.sqrt(
        np.sum(((point - centroid) / std) ** 2)
    )


distance = mahalanobis_from_values(
    point=[1, -3, 4],
    centroid=[0, 0, 0],
    std=[2, 3, 5]
)

print("Mahalanobis distance:", distance)

Mahalanobis distance: 1.374772708486752


First Chunk

In [55]:
first_chunk = X[:CHUNK_SIZE].copy()

print("First chunk shape:", first_chunk.shape)

First chunk shape: (20000, 20)


Farthest First Initalization

In [56]:
def farthest_first_initialization(X, k, random_state=42):
    rng = np.random.default_rng(random_state)

    # Pick the first centroid randomly
    first_index = rng.integers(0, X.shape[0])

    centroid_indices = [first_index]
    centroids = [X[first_index].copy()]

    while len(centroids) < k:
        current_centroids = np.array(centroids)

        distances = np.linalg.norm(
            X[:, np.newaxis, :]
            - current_centroids[np.newaxis, :, :],
            axis=2
        )

        # Distance to nearest already-selected centroid
        min_distances = distances.min(axis=1)

        # Prevent selecting the same point again
        min_distances[centroid_indices] = -1

        next_index = np.argmax(min_distances)

        centroid_indices.append(next_index)
        centroids.append(X[next_index].copy())

    return np.array(centroids)

Initial K means on first chunk (initial seeds to cluster first chunk)

In [57]:
def assign_to_centroids(X, centroids):
    distances = np.linalg.norm(
        X[:, np.newaxis, :]
        - centroids[np.newaxis, :, :],
        axis=2
    )

    return np.argmin(distances, axis=1)

In [58]:
def initial_kmeans(
    X,
    k,
    random_state=42,
    max_iterations=100,
    tolerance=1e-6
):
    centroids = farthest_first_initialization(
        X,
        k,
        random_state
    )

    for iteration in range(max_iterations):

        labels = assign_to_centroids(
            X,
            centroids
        )

        new_centroids = centroids.copy()

        for cluster_id in range(k):
            cluster_points = X[
                labels == cluster_id
            ]

            if len(cluster_points) > 0:
                new_centroids[cluster_id] = (
                    cluster_points.mean(axis=0)
                )

        centroid_shift = np.linalg.norm(
            new_centroids - centroids
        )

        centroids = new_centroids

        if centroid_shift < tolerance:
            break

    # Final assignment
    labels = assign_to_centroids(
        X,
        centroids
    )

    return labels, centroids, iteration + 1


In [59]:
initial_labels, initial_centroids, iterations = initial_kmeans(
    first_chunk,
    k=K,
    random_state=RANDOM_STATE
)

print("Initialization iterations:", iterations)

unique, counts = np.unique(
    initial_labels,
    return_counts=True
)

print("\nInitial cluster sizes:")

for cluster_id, count in zip(unique, counts):
    print(
        f"Cluster {cluster_id}: "
        f"{count:,} points"
    )

Initialization iterations: 49

Initial cluster sizes:
Cluster 0: 6,929 points
Cluster 1: 2,955 points
Cluster 2: 148 points
Cluster 3: 9,968 points


Convert clusters to DS

In [60]:
DS = {}

for cluster_id in range(K):

    cluster_points = first_chunk[
        initial_labels == cluster_id
    ]

    if len(cluster_points) > 0:
        DS[cluster_id] = BFRSummary(
            cluster_points
        )

Inspect DS

In [61]:
print("Number of DS clusters:", len(DS))

for cluster_id, summary in DS.items():

    print(f"\nCluster {cluster_id}")
    print("--------------------")

    print("N:", summary.N)

    print(
        "Centroid first 5 dimensions:",
        summary.centroid()[:5]
    )

    print(
        "Std first 5 dimensions:",
        summary.std()[:5]
    )

Number of DS clusters: 4

Cluster 0
--------------------
N: 6929
Centroid first 5 dimensions: [0.00176218 0.0052441  0.02640289 0.05041306 0.0153934 ]
Std first 5 dimensions: [0.01519601 0.02164452 0.03312554 0.05579998 0.02645809]

Cluster 1
--------------------
N: 2955
Centroid first 5 dimensions: [0.00808645 0.00677053 0.0279888  0.21517419 0.03112538]
Std first 5 dimensions: [0.03184516 0.03693266 0.04707777 0.1734979  0.05191721]

Cluster 2
--------------------
N: 148
Centroid first 5 dimensions: [0.44341021 0.00219408 0.01635915 0.116908   0.01452184]
Std first 5 dimensions: [0.18129322 0.01355596 0.03454188 0.11574805 0.04258398]

Cluster 3
--------------------
N: 9968
Centroid first 5 dimensions: [0.00503002 0.01370163 0.03928906 0.0669811  0.02679427]
Std first 5 dimensions: [0.02127522 0.04375375 0.03874638 0.05209566 0.03422985]


In [62]:
total_ds_points = sum(
    summary.N
    for summary in DS.values()
)

print(
    "Points represented by DS:",
    total_ds_points
)

print(
    "First chunk size:",
    len(first_chunk)
)

assert total_ds_points == len(first_chunk)

Points represented by DS: 20000
First chunk size: 20000


Chunk 2

In [63]:
chunk_2 = X[CHUNK_SIZE:2 * CHUNK_SIZE].copy()

print("Chunk 2 shape:", chunk_2.shape)

Chunk 2 shape: (20000, 20)


Process Chunk 2

In [64]:
def mahalanobis_distance(point, summary, epsilon=1e-10):
    point = np.asarray(point, dtype=float)

    centroid = summary.centroid()
    std = summary.std()

    # Prevent division by zero for dimensions
    # that currently have zero variance
    safe_std = np.maximum(std, epsilon)

    normalized_difference = (
        point - centroid
    ) / safe_std

    distance = np.sqrt(
        np.sum(normalized_difference ** 2)
    )

    return distance


In [65]:
def nearest_ds_cluster(point, DS):
    best_cluster = None
    best_distance = np.inf

    for cluster_id, summary in DS.items():

        distance = mahalanobis_distance(
            point,
            summary
        )

        if distance < best_distance:
            best_distance = distance
            best_cluster = cluster_id

    return best_cluster, best_distance

In [66]:
unassigned_points = []

assigned_count = 0

start_time = time.perf_counter()

for point in chunk_2:

    cluster_id, distance = nearest_ds_cluster(
        point,
        DS
    )

    if distance < MAHALANOBIS_THRESHOLD:

        DS[cluster_id].add_point(point)

        assigned_count += 1

    else:

        unassigned_points.append(
            point.copy()
        )

runtime_chunk_2 = (
    time.perf_counter() - start_time
)

unassigned_points = np.asarray(
    unassigned_points
)

In [67]:
test_point = chunk_2[0]

cluster_id, distance = nearest_ds_cluster(
    test_point,
    DS
)

print("Nearest DS cluster:", cluster_id)
print("Mahalanobis distance:", distance)

Nearest DS cluster: 1
Mahalanobis distance: 2.161278672993768


In [68]:
print("\nUpdated DS sizes:")

for cluster_id, summary in DS.items():
    print(
        f"Cluster {cluster_id}: "
        f"N = {summary.N:,}"
    )

total_ds_points = sum(
    summary.N
    for summary in DS.values()
)

print(
    "\nTotal points represented by DS:",
    f"{total_ds_points:,}"
)

assert total_ds_points == (
    CHUNK_SIZE + assigned_count
)


Updated DS sizes:
Cluster 0: N = 10,326
Cluster 1: N = 6,497
Cluster 2: N = 292
Cluster 3: N = 15,556

Total points represented by DS: 32,671


CS & RS

In [69]:
CS = {}
RS = []

print("Initial number of CS clusters:", len(CS))
print("Initial number of RS points:", len(RS))

Initial number of CS clusters: 0
Initial number of RS points: 0


In [70]:
def build_cs_and_rs(
    rejected_points,
    k,
    random_state=42,
    overclustering_factor=5
):
    rejected_points = np.asarray(
        rejected_points,
        dtype=float
    ).reshape(-1, X.shape[1])

    number_of_points = len(rejected_points)

    # No rejected points
    if number_of_points == 0:
        return {}, []

    # A single point cannot form a CS cluster
    if number_of_points == 1:
        return {}, [rejected_points[0].copy()]

    # Over-cluster the rejected points
    number_of_clusters = min(
        overclustering_factor * k,
        number_of_points
    )

    labels, centroids, iterations = initial_kmeans(
        rejected_points,
        k=number_of_clusters,
        random_state=random_state,
        max_iterations=100
    )

    new_cs = {}
    new_rs = []

    next_cs_id = 0

    for cluster_id in range(number_of_clusters):

        cluster_points = rejected_points[
            labels == cluster_id
        ]

        # Multi-point groups become CS clusters
        if len(cluster_points) > 1:

            new_cs[next_cs_id] = BFRSummary(
                cluster_points
            )

            next_cs_id += 1

        # Singleton groups remain in RS
        elif len(cluster_points) == 1:

            new_rs.append(
                cluster_points[0].copy()
            )

    return new_cs, new_rs

In [71]:
CS, RS = build_cs_and_rs(
    rejected_points=unassigned_points,
    k=K,
    random_state=RANDOM_STATE,
    overclustering_factor=5
)

cs_point_count = sum(
    summary.N
    for summary in CS.values()
)

rs_point_count = len(RS)

print("Number of CS clusters:", len(CS))
print("Number of RS points:", rs_point_count)
print("Points represented by CS:", cs_point_count)
print(
    "Total rejected points accounted for:",
    cs_point_count + rs_point_count
)

assert (
    cs_point_count + rs_point_count
    == len(unassigned_points)
)

Number of CS clusters: 20
Number of RS points: 0
Points represented by CS: 7329
Total rejected points accounted for: 7329


Inspect RS & CS

In [72]:
print("CS and RS results")
print("-----------------")

print("Number of CS clusters:", len(CS))
print("Number of RS points:", len(RS))

cs_point_count = sum(
    summary.N
    for summary in CS.values()
)

print("Points represented by CS:", cs_point_count)
print("Points retained individually in RS:", len(RS))
print(
    "Total rejected points accounted for:",
    cs_point_count + len(RS)
)

CS and RS results
-----------------
Number of CS clusters: 20
Number of RS points: 0
Points represented by CS: 7329
Points retained individually in RS: 0
Total rejected points accounted for: 7329


In [73]:
print("Chunk 2 size:", len(chunk_2))
print("Assigned to DS:", assigned_count)
print("Rejected from DS:", len(unassigned_points))
print("Threshold:", MAHALANOBIS_THRESHOLD)

Chunk 2 size: 20000
Assigned to DS: 12671
Rejected from DS: 7329
Threshold: 4.0


In [74]:
assert (
    cs_point_count + len(RS)
    == len(unassigned_points)
)

print("\nAll rejected points were assigned to CS or RS.")


All rejected points were assigned to CS or RS.


Chunk 3

In [75]:
chunk_3 = X[
    2 * CHUNK_SIZE:
    3 * CHUNK_SIZE
].copy()

print("Chunk 3 shape:", chunk_3.shape)

Chunk 3 shape: (20000, 20)


In [76]:
ds_before = sum(
    summary.N
    for summary in DS.values()
)

cs_before = sum(
    summary.N
    for summary in CS.values()
)

rs_before = len(RS)

print("Before Chunk 3:")
print("DS points:", ds_before)
print("CS points:", cs_before)
print("RS points:", rs_before)

Before Chunk 3:
DS points: 32671
CS points: 7329
RS points: 0


In [77]:
chunk_3_rejected = []
chunk_3_ds_assigned = 0

start_time = time.perf_counter()

for point in chunk_3:

    cluster_id, distance = nearest_ds_cluster(
        point,
        DS
    )

    if distance < MAHALANOBIS_THRESHOLD:

        DS[cluster_id].add_point(point)
        chunk_3_ds_assigned += 1

    else:

        chunk_3_rejected.append(
            point.copy()
        )

chunk_3_rejected = np.asarray(
    chunk_3_rejected,
    dtype=float
).reshape(-1, X.shape[1])

In [78]:
if len(RS) > 0:

    old_rs_array = np.asarray(
        RS,
        dtype=float
    ).reshape(-1, X.shape[1])

    points_to_recluster = np.vstack([
        old_rs_array,
        chunk_3_rejected
    ])

else:

    points_to_recluster = (
        chunk_3_rejected.copy()
    )

print(
    "Points to recluster:",
    len(points_to_recluster)
)

Points to recluster: 8704


In [79]:
new_CS, new_RS = build_cs_and_rs(
    rejected_points=points_to_recluster,
    k=K,
    random_state=RANDOM_STATE,
    overclustering_factor=5
)

In [80]:
next_cs_id = (
    max(CS.keys()) + 1
    if len(CS) > 0
    else 0
)

for _, summary in new_CS.items():

    CS[next_cs_id] = summary

    next_cs_id += 1

RS = new_RS

In [81]:
ds_count = sum(
    summary.N
    for summary in DS.values()
)

cs_count = sum(
    summary.N
    for summary in CS.values()
)

rs_count = len(RS)

print("\nBFR state after Chunk 3")
print("-----------------------")

print("DS points:", ds_count)
print("Number of CS clusters:", len(CS))
print("CS points:", cs_count)
print("RS points:", rs_count)

print(
    "\nChunk 3 assigned directly to DS:",
    chunk_3_ds_assigned
)

print(
    "Chunk 3 rejected by DS:",
    len(chunk_3_rejected)
)


BFR state after Chunk 3
-----------------------
DS points: 43967
Number of CS clusters: 40
CS points: 16033
RS points: 0

Chunk 3 assigned directly to DS: 11296
Chunk 3 rejected by DS: 8704


Merging Clusters

In [82]:
def combine_summaries(summary1, summary2):
    combined = BFRSummary.__new__(BFRSummary)

    combined.N = summary1.N + summary2.N
    combined.SUM = summary1.SUM + summary2.SUM
    combined.SUMSQ = summary1.SUMSQ + summary2.SUMSQ

    return combined

In [83]:
def should_merge_cs(
    summary1,
    summary2,
    variance_factor=1.5
):
    combined = combine_summaries(
        summary1,
        summary2
    )

    variance1 = np.mean(
        summary1.variance()
    )

    variance2 = np.mean(
        summary2.variance()
    )

    combined_variance = np.mean(
        combined.variance()
    )

    # Weighted average variance of the
    # two clusters before merging
    original_variance = (
        summary1.N * variance1
        + summary2.N * variance2
    ) / (
        summary1.N
        + summary2.N
    )

    # Handle extremely small/zero variance
    if original_variance < 1e-12:
        return combined_variance < 1e-12

    return (
        combined_variance
        <= variance_factor * original_variance
    )

In [84]:
def merge_cs_clusters(
    CS,
    variance_factor=1.5
):
    CS = CS.copy()

    total_merges = 0

    while True:

        cluster_ids = list(CS.keys())

        merge_found = False

        for i in range(len(cluster_ids)):

            for j in range(i + 1, len(cluster_ids)):

                id1 = cluster_ids[i]
                id2 = cluster_ids[j]

                summary1 = CS[id1]
                summary2 = CS[id2]

                if should_merge_cs(
                    summary1,
                    summary2,
                    variance_factor
                ):

                    # Merge id2 into id1
                    CS[id1] = combine_summaries(
                        summary1,
                        summary2
                    )

                    del CS[id2]

                    total_merges += 1
                    merge_found = True

                    break

            if merge_found:
                break

        # Nothing else can be merged
        if not merge_found:
            break

    return CS, total_merges

In [85]:
cs_clusters_before = len(CS)

cs_points_before = sum(
    summary.N
    for summary in CS.values()
)

print(
    "CS clusters before merging:",
    cs_clusters_before
)

print(
    "CS points before merging:",
    cs_points_before
)

CS clusters before merging: 40
CS points before merging: 16033


In [86]:
CS, number_of_merges = merge_cs_clusters(
    CS,
    variance_factor=1.5
)

print(
    "Number of CS merges:",
    number_of_merges
)

Number of CS merges: 39


In [87]:
cs_clusters_after = len(CS)

cs_points_after = sum(
    summary.N
    for summary in CS.values()
)

print(
    "CS clusters after merging:",
    cs_clusters_after
)

print(
    "CS points after merging:",
    cs_points_after
)

CS clusters after merging: 1
CS points after merging: 16033


In [88]:
assert cs_points_before == cs_points_after

print(
    "CS merging preserved all points."
)

CS merging preserved all points.


In [89]:
ds_count = sum(
    summary.N
    for summary in DS.values()
)

cs_count = sum(
    summary.N
    for summary in CS.values()
)

rs_count = len(RS)

print("BFR state after CS merging")
print("--------------------------")

print(f"DS clusters: {len(DS)}")
print(f"DS points: {ds_count:,}")

print(f"\nCS clusters: {len(CS)}")
print(f"CS points: {cs_count:,}")

print(f"\nRS points: {rs_count:,}")

print(
    "\nTotal represented:",
    f"{ds_count + cs_count + rs_count:,}"
)

BFR state after CS merging
--------------------------
DS clusters: 4
DS points: 43,967

CS clusters: 1
CS points: 16,033

RS points: 0

Total represented: 60,000


Now that we know the individual chunks work, we'll automatically, and programitcally, do the remaining

In [90]:
def process_chunk(
    chunk,
    DS,
    CS,
    RS,
    k,
    mahalanobis_threshold=4.0,
    variance_factor=1.5,
    overclustering_factor=5,
    random_state=42
):
    start_time = time.perf_counter()

    # ---------------------------------------------
    # 1. Record state before processing
    # ---------------------------------------------

    ds_before = sum(
        summary.N
        for summary in DS.values()
    )

    cs_before = sum(
        summary.N
        for summary in CS.values()
    )

    rs_before = len(RS)

    # ---------------------------------------------
    # 2. Attempt to assign new points to DS
    # ---------------------------------------------

    rejected_points = []
    assigned_to_ds = 0

    for point in chunk:

        cluster_id, distance = nearest_ds_cluster(
            point,
            DS
        )

        if distance < mahalanobis_threshold:

            DS[cluster_id].add_point(point)

            assigned_to_ds += 1

        else:

            rejected_points.append(
                point.copy()
            )

    rejected_points = np.asarray(
        rejected_points,
        dtype=float
    ).reshape(-1, X.shape[1])

    # ---------------------------------------------
    # 3. Combine rejected points with old RS
    # ---------------------------------------------

    if len(RS) > 0:

        old_rs = np.asarray(
            RS,
            dtype=float
        ).reshape(-1, X.shape[1])

        points_to_recluster = np.vstack([
            old_rs,
            rejected_points
        ])

    else:

        points_to_recluster = (
            rejected_points.copy()
        )

    # ---------------------------------------------
    # 4. Build new CS and RS groups
    # ---------------------------------------------

    new_CS, new_RS = build_cs_and_rs(
        rejected_points=points_to_recluster,
        k=k,
        random_state=random_state,
        overclustering_factor=overclustering_factor
    )

    # ---------------------------------------------
    # 5. Add new CS summaries to existing CS
    # ---------------------------------------------

    next_cs_id = (
        max(CS.keys()) + 1
        if len(CS) > 0
        else 0
    )

    for summary in new_CS.values():

        CS[next_cs_id] = summary
        next_cs_id += 1

    # New singleton points become the RS
    RS = new_RS

    # ---------------------------------------------
    # 6. Merge compatible CS miniclusters
    # ---------------------------------------------

    CS, number_of_merges = merge_cs_clusters(
        CS,
        variance_factor=variance_factor
    )

    # ---------------------------------------------
    # 7. Calculate final statistics
    # ---------------------------------------------

    ds_after = sum(
        summary.N
        for summary in DS.values()
    )

    cs_after = sum(
        summary.N
        for summary in CS.values()
    )

    rs_after = len(RS)

    runtime = (
        time.perf_counter()
        - start_time
    )

    stats = {
        "chunk_size": len(chunk),

        "assigned_to_ds": assigned_to_ds,
        "rejected_by_ds": len(rejected_points),

        "ds_points_before": ds_before,
        "ds_points_after": ds_after,

        "cs_points_before": cs_before,
        "cs_points_after": cs_after,

        "rs_points_before": rs_before,
        "rs_points_after": rs_after,

        "cs_clusters": len(CS),
        "cs_merges": number_of_merges,

        "runtime": runtime
    }

    return DS, CS, RS, stats

In [91]:
bfr_history = []

start_index = 3 * CHUNK_SIZE

chunk_number = 4

total_start_time = time.perf_counter()

for start in range(
    start_index,
    len(X),
    CHUNK_SIZE
):
    end = min(
        start + CHUNK_SIZE,
        len(X)
    )

    chunk = X[start:end]

    print(
        f"\nProcessing Chunk {chunk_number} "
        f"(rows {start:,} to {end - 1:,})"
    )

    DS, CS, RS, stats = process_chunk(
        chunk=chunk,
        DS=DS,
        CS=CS,
        RS=RS,
        k=K,
        mahalanobis_threshold=MAHALANOBIS_THRESHOLD,
        variance_factor=1.5,
        overclustering_factor=5,
        random_state=RANDOM_STATE
    )

    stats["chunk_number"] = chunk_number
    stats["start_row"] = start
    stats["end_row"] = end

    bfr_history.append(stats)

    print(
        f"  Assigned to DS: "
        f"{stats['assigned_to_ds']:,}"
    )

    print(
        f"  Rejected by DS: "
        f"{stats['rejected_by_ds']:,}"
    )

    print(
        f"  DS points: "
        f"{stats['ds_points_after']:,}"
    )

    print(
        f"  CS clusters: "
        f"{stats['cs_clusters']:,}"
    )

    print(
        f"  CS points: "
        f"{stats['cs_points_after']:,}"
    )

    print(
        f"  RS points: "
        f"{stats['rs_points_after']:,}"
    )

    print(
        f"  CS merges: "
        f"{stats['cs_merges']:,}"
    )

    print(
        f"  Runtime: "
        f"{stats['runtime']:.4f} seconds"
    )

    chunk_number += 1


total_bfr_runtime = (
    time.perf_counter()
    - total_start_time
)

print(
    f"\nRemaining chunk-processing runtime: "
    f"{total_bfr_runtime:.4f} seconds"
)


Processing Chunk 4 (rows 60,000 to 79,999)
  Assigned to DS: 10,291
  Rejected by DS: 9,709
  DS points: 54,258
  CS clusters: 1
  CS points: 25,742
  RS points: 0
  CS merges: 20
  Runtime: 2.5677 seconds

Processing Chunk 5 (rows 80,000 to 99,999)
  Assigned to DS: 9,525
  Rejected by DS: 10,475
  DS points: 63,783
  CS clusters: 1
  CS points: 36,216
  RS points: 1
  CS merges: 19
  Runtime: 3.6230 seconds

Processing Chunk 6 (rows 100,000 to 119,999)
  Assigned to DS: 8,831
  Rejected by DS: 11,169
  DS points: 72,614
  CS clusters: 1
  CS points: 47,385
  RS points: 1
  CS merges: 19
  Runtime: 3.1103 seconds

Processing Chunk 7 (rows 120,000 to 139,999)
  Assigned to DS: 8,307
  Rejected by DS: 11,693
  DS points: 80,921
  CS clusters: 1
  CS points: 59,078
  RS points: 1
  CS merges: 19
  Runtime: 3.7571 seconds

Processing Chunk 8 (rows 140,000 to 159,999)
  Assigned to DS: 8,037
  Rejected by DS: 11,963
  DS points: 88,958
  CS clusters: 1
  CS points: 71,041
  RS points: 1
 

In [92]:
bfr_history_df = pd.DataFrame(
    bfr_history
)

bfr_history_df

,chunk_size,assigned_to_ds,rejected_by_ds,ds_points_before,ds_points_after,cs_points_before,cs_points_after,rs_points_before,rs_points_after,cs_clusters,cs_merges,runtime,chunk_number,start_row,end_row
0,20000,10291,9709,43967,54258,16033,25742,0,0,1,20,2.567700,4,60000,80000
1,20000,9525,10475,54258,63783,25742,36216,0,1,1,19,3.622989,5,80000,100000
2,20000,8831,11169,63783,72614,36216,47385,1,1,1,19,3.110310,6,100000,120000
3,20000,8307,11693,72614,80921,47385,59078,1,1,1,19,3.757112,7,120000,140000
4,20000,8037,11963,80921,88958,59078,71041,1,1,1,19,2.936300,8,140000,160000
5,20000,7718,12282,88958,96676,71041,83323,1,1,1,19,3.433131,9,160000,180000
6,20000,7473,12527,96676,104149,83323,95850,1,1,1,19,2.783660,10,180000,200000
7,6209,2251,3958,104149,106400,95850,99808,1,1,1,19,0.948484,11,200000,206209


In [93]:
final_ds_count = sum(
    summary.N
    for summary in DS.values()
)

final_cs_count = sum(
    summary.N
    for summary in CS.values()
)

final_rs_count = len(RS)

total_accounted = (
    final_ds_count
    + final_cs_count
    + final_rs_count
)

print("BFR State Before Final Cleanup")
print("------------------------------")

print(f"DS clusters: {len(DS)}")
print(f"DS points: {final_ds_count:,}")

print(f"\nCS clusters: {len(CS)}")
print(f"CS points: {final_cs_count:,}")

print(f"\nRS points: {final_rs_count:,}")

print(
    f"\nTotal points represented: "
    f"{total_accounted:,}"
)

print(
    f"Expected total users: "
    f"{len(X):,}"
)

BFR State Before Final Cleanup
------------------------------
DS clusters: 4
DS points: 106,400

CS clusters: 1
CS points: 99,808

RS points: 1

Total points represented: 206,209
Expected total users: 206,209


Final stage

In [94]:
def nearest_ds_for_summary(cs_summary, DS):
    point = cs_summary.centroid()

    best_cluster = None
    best_distance = np.inf

    for cluster_id, ds_summary in DS.items():

        distance = mahalanobis_distance(
            point,
            ds_summary
        )

        if distance < best_distance:
            best_distance = distance
            best_cluster = cluster_id

    return best_cluster, best_distance

In [95]:
def finalize_cs(
    DS,
    CS,
    threshold=4.0
):
    remaining_CS = {}

    merged_cs_clusters = 0
    merged_cs_points = 0

    for cs_id, cs_summary in CS.items():

        cluster_id, distance = nearest_ds_for_summary(
            cs_summary,
            DS
        )

        if distance < threshold:

            DS[cluster_id].merge(
                cs_summary
            )

            merged_cs_clusters += 1
            merged_cs_points += cs_summary.N

        else:

            remaining_CS[cs_id] = cs_summary

    return (
        DS,
        remaining_CS,
        merged_cs_clusters,
        merged_cs_points
    )

In [96]:
DS, CS, merged_cs_clusters, merged_cs_points = finalize_cs(
    DS,
    CS,
    threshold=MAHALANOBIS_THRESHOLD
)

print("CS finalization")
print("---------------")
print(
    "CS clusters merged into DS:",
    merged_cs_clusters
)
print(
    "CS points merged into DS:",
    merged_cs_points
)
print(
    "CS clusters remaining:",
    len(CS)
)

CS finalization
---------------
CS clusters merged into DS: 1
CS points merged into DS: 99808
CS clusters remaining: 0


In [97]:
def finalize_rs(
    DS,
    RS,
    threshold=4.0
):
    outliers = []

    assigned_rs_points = 0

    for point in RS:

        cluster_id, distance = nearest_ds_cluster(
            point,
            DS
        )

        if distance < threshold:

            DS[cluster_id].add_point(
                point
            )

            assigned_rs_points += 1

        else:

            outliers.append(
                point.copy()
            )

    return (
        DS,
        outliers,
        assigned_rs_points
    )

In [98]:
DS, rs_outliers, assigned_rs_points = finalize_rs(
    DS,
    RS,
    threshold=MAHALANOBIS_THRESHOLD
)

print("RS finalization")
print("---------------")

print(
    "RS points merged into DS:",
    assigned_rs_points
)

print(
    "RS points remaining as outliers:",
    len(rs_outliers)
)

RS finalization
---------------
RS points merged into DS: 0
RS points remaining as outliers: 1


In [99]:
cs_outlier_points = sum(
    summary.N
    for summary in CS.values()
)

rs_outlier_points = len(
    rs_outliers
)

total_outliers = (
    cs_outlier_points
    + rs_outlier_points
)

print("Final outliers")
print("--------------")

print(
    "Points in unmerged CS:",
    cs_outlier_points
)

print(
    "Unmerged RS points:",
    rs_outlier_points
)

print(
    "Total outlier points:",
    total_outliers
)

Final outliers
--------------
Points in unmerged CS: 0
Unmerged RS points: 1
Total outlier points: 1


In [100]:
final_ds_count = sum(
    summary.N
    for summary in DS.values()
)

print("Final BFR Results")
print("-----------------")

print(
    "Final DS clusters:",
    len(DS)
)

for cluster_id, summary in DS.items():

    print(
        f"Cluster {cluster_id}: "
        f"{summary.N:,} points"
    )

print(
    "\nPoints assigned to final clusters:",
    f"{final_ds_count:,}"
)

print(
    "Outlier points:",
    f"{total_outliers:,}"
)

print(
    "Total accounted for:",
    f"{final_ds_count + total_outliers:,}"
)

print(
    "Expected:",
    f"{len(X):,}"
)

Final BFR Results
-----------------
Final DS clusters: 4
Cluster 0: 27,915 points
Cluster 1: 30,515 points
Cluster 2: 1,399 points
Cluster 3: 146,379 points

Points assigned to final clusters: 206,208
Outlier points: 1
Total accounted for: 206,209
Expected: 206,209


In [101]:
bfr_centroids = np.vstack([
    DS[cluster_id].centroid()
    for cluster_id in sorted(DS.keys())
])

bfr_centroid_df = pd.DataFrame(
    bfr_centroids,
    columns=feature_columns
)

bfr_centroid_df.index.name = "cluster"

bfr_centroid_df

,alcohol,babies,bakery,beverages,breakfast,bulk,canned_goods,dairy_eggs,deli,dry_goods_pasta,frozen,household,international,meat_seafood,other,pantry,personal_care,pets,produce,snacks
cluster,,,,,,,,,,,,,,,,,,,,
0,0.000473,0.002451,0.024429,0.045492,0.012990,0.000919,0.043061,0.129968,0.027416,0.027714,0.052408,0.011441,0.009394,0.025182,0.000555,0.079930,0.008021,0.000213,0.439656,0.058287
1,0.002295,0.003038,0.033063,0.131526,0.029997,0.000106,0.021949,0.134367,0.028706,0.018684,0.079000,0.055930,0.004646,0.012440,0.001162,0.076752,0.034904,0.001936,0.177012,0.152486
2,0.341359,0.000275,0.013413,0.170838,0.014745,0.000000,0.003032,0.075663,0.015596,0.000377,0.028462,0.040413,0.003057,0.002155,0.004317,0.051870,0.004635,0.000331,0.090798,0.138664
3,0.007330,0.012608,0.034320,0.080363,0.024227,0.001278,0.041015,0.138123,0.031486,0.033501,0.078285,0.038072,0.011846,0.024369,0.001911,0.084425,0.023694,0.003744,0.240862,0.088540


In [102]:
for cluster_id in bfr_centroid_df.index:

    print(f"\nBFR Cluster {cluster_id}")
    print("--------------------")

    print(
        bfr_centroid_df
        .loc[cluster_id]
        .sort_values(ascending=False)
        .head(5)
    )


BFR Cluster 0
--------------------
produce       0.439656
dairy_eggs    0.129968
pantry        0.079930
snacks        0.058287
frozen        0.052408
Name: 0, dtype: float64

BFR Cluster 1
--------------------
produce       0.177012
snacks        0.152486
dairy_eggs    0.134367
beverages     0.131526
frozen        0.079000
Name: 1, dtype: float64

BFR Cluster 2
--------------------
alcohol       0.341359
beverages     0.170838
snacks        0.138664
produce       0.090798
dairy_eggs    0.075663
Name: 2, dtype: float64

BFR Cluster 3
--------------------
produce       0.240862
dairy_eggs    0.138123
snacks        0.088540
pantry        0.084425
beverages     0.080363
Name: 3, dtype: float64


What has been implemented 
Department-level customer vectors
               ↓
         first data chunk
               ↓
       initialize K clusters
               ↓
             DS
       N / SUM / SUMSQ
               ↓
      read subsequent chunk
               ↓
     Mahalanobis distance
          /           \
     close           not close
       ↓                 ↓
      DS            old RS + points
                         ↓
                    over-cluster
                     /        \
                    CS        RS
                     ↓
                 merge CS
                     ↓
              next data chunk
                     ↓
              final cleanup
            /        |       \
           DS       CS       RS
           ↑         |        |
           └──── eligible ────┘
                     ↓
               final DS
                  +
               outliers